# Context Budget Lab

### Source : CONTEXT_FUNDAMENTALS_1/context_budget_lab_1.ipynb
### Context confusion illustration

## Setup

This cell reads from a `.env` file in the project root:
- `DATABRICKS_HOST` and `DATABRICKS_TOKEN` — required.
- The model used is `databricks-llama-4-maverick`, a pay-per-token endpoint on
  Databricks Foundation Model APIs (no provisioned-throughput deployment needed). It has a
  128,000-token context window and supports function calling with the `tools` / `tool_choice`
  parameters, per Databricks' own function-calling documentation.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set — check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set — check your .env file"

DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
MODEL_ENDPOINT = "databricks-llama-4-maverick"
MODEL_CONTEXT_WINDOW = 128_000  # databricks-llama-4-maverick, per Databricks documentation

In [ ]:
import json
import random
import tiktoken

random.seed(7)

# Token counting note: Llama models use their own tokenizer (not OpenAI's), so tiktoken here is a
# stand-in for order-of-magnitude comparisons, not an exact count. If you have Hugging Face access
# and have accepted Meta's license, swap in the real tokenizer for accurate numbers:

try:
    enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(enc.encode(text))
except Exception as e:
    # tiktoken fetches its encoding file from a remote blob store on first use; this fails behind
    # some corporate proxies/firewalls. Falls back to an approximate counter so the lab still runs.
    print(f"tiktoken encoding unavailable ({e.__class__.__name__}); using an approximate fallback counter.")
    import re
    def count_tokens(text: str) -> int:
        return len(re.findall(r"\w+|[^\w\s]", text))

## Section 1 — Build the 55 MCP tool schemas (5 used, 50 noise)
Same structure as the contract agent: 5 tools the tool chain actually calls, 50 tools from
unrelated domains that are loaded but never needed this session.

Each schema now carries 5 extra metadata parameters (`request_id`, `caller_context`,
`priority_level`, `environment`, `requested_by`) on top of its 3 functional parameters — closer to
how a real production tool catalog looks, where tracing/auth/routing metadata usually outweighs the
actual functional arguments. The description was already a minority of a schema's tokens (most of
the weight was already in `parameters`); adding metadata parameters pushes that further rather than
padding the description text itself.

In [ ]:
# Common metadata parameters every tool schema carries, in addition to its functional ones.
# This is what makes real tool catalogs heavier than a minimal "3 functional params" schema.
METADATA_PARAMS = {
    "request_id": {"type": "string", "description": "A unique identifier for tracing this call across the MLflow trace log."},
    "caller_context": {"type": "string", "description": "Free-text context about which upstream workflow step triggered this call."},
    "priority_level": {"type": "string", "description": "The urgency of this call: low, normal, or high, used for internal queueing."},
    "environment": {"type": "string", "description": "The deployment environment this call is being made from, e.g. staging or production."},
    "requested_by": {"type": "string", "description": "The identifier of the user or service account on whose behalf this call is made."},
}

def make_tool_schema(name, description, n_params=3):
    properties = {
        f"param_{i}": {"type": "string", "description": f"Parameter {i} for {name}."}
        for i in range(n_params)
    }
    properties.update(METADATA_PARAMS)  # +5 properties per tool, 8 total, still under the 16-key limit
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
            },
        },
    }

In [ ]:
CORE_TOOLS = [
    make_tool_schema("extract_clauses", "Extracts clauses and their boundaries from a contract."),
    make_tool_schema("retrieve_refs", "Retrieves the regulatory references linked to a clause."),
    make_tool_schema("check_compliance", "Checks a clause's compliance against retrieved references."),
    make_tool_schema("generate_summary", "Generates a structured risk summary for the contract."),
    make_tool_schema("log_trace", "Logs an MLflow trace entry for the current step."),
]
CORE_TOOL_NAMES = [t["function"]["name"] for t in CORE_TOOLS]



In [ ]:
NOISE_DOMAINS = ["billing", "crm", "hr", "payroll", "shipping", "support", "marketing", "inventory", "sales", "legal_research"]
# Domains whose natural-language descriptions plausibly overlap with contract-review vocabulary.
OVERLAP_TERMS = {
    "billing": "financial liability",
    "legal_research": "regulatory compliance",
    "hr": "contractual obligation",
}

NOISE_TOOLS = []
for i in range(50):
    domain = NOISE_DOMAINS[i % len(NOISE_DOMAINS)]
    name = f"{domain}_action_{i}"
    if domain in OVERLAP_TERMS:
        desc = f"Handles {domain} operations, including checks related to {OVERLAP_TERMS[domain]}."
    else:
        desc = f"Handles routine {domain} operations unrelated to contract review."
    NOISE_TOOLS.append(make_tool_schema(name, desc))

In [ ]:

ALL_55_TOOLS = CORE_TOOLS + NOISE_TOOLS
assert len(ALL_55_TOOLS) == 55

print(f"Core tools actually used by the tool chain: {len(CORE_TOOLS)}")
print(f"Noise tools loaded but never needed: {len(NOISE_TOOLS)}")

# Weight diagnostic: how much of one schema's tokens is the description vs. the rest?
_example = CORE_TOOLS[2]  # check_compliance
_desc_tokens = count_tokens(_example["function"]["description"])
_total_tokens = count_tokens(json.dumps(_example))
print(f"\nExample schema weight — '{_example['function']['name']}': "
      f"description = {_desc_tokens} tokens, full schema = {_total_tokens} tokens "
      f"({_desc_tokens/_total_tokens:.0%} of the schema is the description)")

## Section 2 — Build 10 test questions, each with metadata

10 questions (2 per core tool) so a live run stays fast — each question triggers
two model calls in Section 4. 

In [ ]:
QUESTION_TEMPLATES = {
    "extract_clauses": [
        "Identify the clause boundaries in section {n} of this contract.",
        "Split this agreement into its individual clauses starting at paragraph {n}.",
    ],
    "retrieve_refs": [
        "What regulations apply to the indemnification clause in section {n}?",
        "Pull the regulatory references linked to clause {n}.",
    ],
    "check_compliance": [
        "Does clause {n} on limitation of liability comply with the retrieved references?",
        "Cross-check clause {n} against the applicable regulations for compliance.",
    ],
    "generate_summary": [
        "Produce a risk summary for this contract covering clauses 1 through {n}.",
        "Summarize the overall compliance risk of the document up to clause {n}.",
    ],
    "log_trace": [
        "Record a trace entry for the review step on clause {n}.",
        "Log the outcome of the compliance check for clause {n}.",
    ],
}


In [ ]:
questions = []
qid = 0
for tool_name, templates in QUESTION_TEMPLATES.items():
    for i in range(2):  # 5 tools * 2 = 10 questions
        template = templates[i % len(templates)]
        prompt = template.format(n=i + 1)
        questions.append({
            "question_id": f"Q{qid:03d}",
            "prompt": prompt,
            "expected_tool": tool_name,
            "domain": "contract_review",
            "is_core_tool_question": True,
            "overlap_risk": "liability" in prompt.lower() or "compliance" in prompt.lower() or "obligation" in prompt.lower(),
            "prompt_tokens": count_tokens(prompt),
        })
        qid += 1

assert len(questions) == 10

print(f"Generated {len(questions)} metadata-tagged test questions.")
print(json.dumps(questions[:2], indent=2))

## Section 3 — Call the model

One function, one path: every call goes live against `MODEL_ENDPOINT`. There's no simulated mode
here — with only 10 questions and 2 passes, a live run is fast enough that a synthetic fallback
isn't needed.

In [ ]:
def call_model(prompt, tools):
    # OpenAI-compatible call against a Databricks Foundation Model API serving endpoint.
    from openai import OpenAI
    client = OpenAI(api_key=DATABRICKS_TOKEN, base_url=f"{DATABRICKS_HOST}/serving-endpoints")
    response = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice="auto",
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        return msg.tool_calls[0].function.name
    return None

## Section 4 — Three passes: core + 15 noise, core + only the misused tools, core alone

A reusable test-runner function first (`run_tool_selection_test`), then three passes with it:

- **Pass 1 — core + 50 noise (55 tools total)**: This checks whether confusion shows up.
- **Pass 2 — core only (5 tools)**: the clean baseline, no noise at all.


In [ ]:
def get_tool_schema_by_name(name, all_tools):
    return next(t for t in all_tools if t["function"]["name"] == name)

def run_tool_selection_test(tools, questions, label=""):
    # Binds a fixed tool list to a call function: run every question against exactly this
    # loadout, printing called-vs-expected for each one, and report overall accuracy.
    if label:
        print(f"--- {label} ({len(tools)} tools) ---")
    results = []
    for q in questions:
        called = call_model(q["prompt"], tools)
        correct = called == q["expected_tool"]
        mark = "✓" if correct else "✗"
        print(f"  [{mark}] {q['question_id']}  expected={q['expected_tool']:<18} called={called}")
        results.append({
            **q,
            "called_tool": called,
            "correct": correct,
        })
    accuracy = sum(r["correct"] for r in results) / len(results)
    return results, accuracy

In [ ]:
# Pass 1 — core (5) + a 50-tool sample of the noise pool
TOOLS_55 = CORE_TOOLS + NOISE_TOOLS
assert len(TOOLS_55) == 55

results_pass1, accuracy_pass1 = run_tool_selection_test(TOOLS_55, questions, label="Pass 1: core + 50 noise")

mistaken_tools = sorted({
    r["called_tool"] for r in results_pass1
    if r["called_tool"] and r["called_tool"] != r["expected_tool"]
})
print(f"\nTools mistakenly called in pass 1: {mistaken_tools if mistaken_tools else '(none)'}")

In [ ]:
# Pass 2 — core only, the clean baseline
results_pass2, accuracy_pass2 = run_tool_selection_test(CORE_TOOLS, questions, label="\nPass 2: core only")

In [ ]:
print(f"\nPass 1 — core + 50 noise (55 tools, {count_tokens(json.dumps(TOOLS_55))} tokens)            : {accuracy_pass1:.1%} accuracy")
print(f"Pass 2 — core only (5 tools, {count_tokens(json.dumps(CORE_TOOLS))} tokens)                    : {accuracy_pass2:.1%} accuracy")

Confusion, per the course's definition, is attention allocated to irrelevant
tokens forcing the model to process noise alongside signal — it is a property of what proportion of
the model's attention gets spent on irrelevant material, not of whether the window has free
capacity left. The three errors don't lead to a peripheral tool (billing, HR, etc.) but rather to another core tool—or result in no call at all—indicating that the issue is a dilution of attention across 55 schemas, rather than a specific lexical trap like the ones encountered in previous attempts.

## Section 5 — Context window budget: both passes, on the window we're actually using

Compare how much of the model's context window each of the three tool loadouts from Section 4
actually consumes.

In [ ]:
tokens_pass1 = count_tokens(json.dumps(TOOLS_55))
tokens_pass2 = count_tokens(json.dumps(CORE_TOOLS))

print(f"Pass 1 (core + 50 noise)     : {tokens_pass1} tokens = {tokens_pass1/MODEL_CONTEXT_WINDOW:.2%} of the window")
print(f"Pass 2 (core only)           : {tokens_pass2} tokens = {tokens_pass2/MODEL_CONTEXT_WINDOW:.2%} of the window")

A 128K window with 50 irrelevant tools still asks the model to consider 55 schemas
before picking one; the noise-to-signal ratio in that decision is unchanged by how much unused
space sits elsewhere in the window. This is exactly why the course's fix for confusion is diclosure.